# 02 · What makes the trajectory features work?

**NFL Big Data Bowl 2026 — Prediction · Feature research completion gate**

The bank contains **7,999 candidates in 20 families**. The central question is whether their signals improve the official metric under the same estimator and honest chronological validation. Follow the evidence from a 64-feature reference through nested additions, wider budgets, family removals, and availability tests.

The reserved 48-game holdout is unscored. Development results are not a leaderboard claim. The final section retains an owner-controlled export; automated review never enables it.

In [ ]:
import io
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import plotly.express as px
from IPython.display import Image, Markdown, display

from nfl_trajectory.research import load_evidence, load_research_report

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
PUBLISHED = ROOT / "docs/results"
feature_summary, selection, evidence_label = load_evidence(ROOT)


def extra_report(local, published):
    path = ROOT / "artifacts" / local
    if not path.is_file():
        path = PUBLISHED / published
    if not path.is_file():
        return None
    result = json.loads(path.read_text())
    if result.get("status") != "passed" or result.get("holdout_evaluation") != "not_run":
        raise ValueError("A completed report with an unscored holdout is required.")
    return result


def static(figure):
    buffer = io.BytesIO()
    figure.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    plt.close(figure)
    display(Image(data=buffer.getvalue()))


def interactive(figure):
    figure.update_layout(template="plotly_white", font={"size": 13})
    display({"application/vnd.plotly.v1+json": json.loads(figure.to_json())}, raw=True)


research = importance = None
if (ROOT / "artifacts/research/report/summary.json").is_file() or (
    PUBLISHED / "feature_research.json"
).is_file():
    research, importance, research_label = load_research_report(ROOT)
    display(Markdown(f"**Research evidence:** {research_label}"))
probe = extra_report("nonlinear_probe/summary.json", "feature_probe.json")
ablation = extra_report("feature_ablation/summary.json", "feature_ablation.json")
budget = extra_report("feature_budget/summary.json", "feature_budget.json")
joint = extra_report("joint_linear/summary.json", "feature_joint.json")
inference = extra_report("research/inference/summary.json", "feature_inference.json")
attribution = extra_report("feature_attribution/summary.json", "feature_attribution.json")
gateway = extra_report("research/gateway/summary.json", "feature_gateway.json")
display(
    pd.DataFrame(
        [
            {"Evidence": name, "Available": value is not None}
            for name, value in [
                ("Candidate research", research),
                ("Fixed estimator comparisons", probe),
                ("Strict removals", ablation),
                ("Width search", budget),
                ("Joint linear fit", joint),
                ("Raw inference", inference),
                ("Wide attribution", attribution),
                ("Gateway", gateway),
            ]
        ]
    )
)

## 1 · Attribute gains to the features with estimator settings fixed

Every nonlinear comparison uses two residual regressors: **100 iterations, depth 4, at most 15 leaves, learning rate 0.07, minimum leaf size 60, L2 penalty 1, and 63 bins**. Early stopping is disabled so a random frame split cannot enter training. The physical baseline is refitted within each training fold.

The landing reference, engineered core, context, representations, and their union use identical rows and settings. Independent noise columns are a negative control and never enter feature selection. Comparing these rows isolates representation gains at this fixed capacity; comparing a linear score with a tree score would also change the estimator.

In [ ]:
if probe:
    scores = pd.DataFrame(probe["models"])
    scores = scores.loc[~scores.model.eq("constant_velocity")].copy()
    reference = scores.loc[scores.model.eq("landing_features"), "coordinate_rmse_yards"].iloc[0]
    scores["feature_gain_percent"] = 100 * (1 - scores.coordinate_rmse_yards / reference)
    display(
        scores[
            [
                "model",
                "feature_count",
                "coordinate_rmse_yards",
                "feature_gain_percent",
                "delta_vs_landing_ci95",
            ]
        ].round(5)
    )
    interactive(
        px.bar(
            scores,
            x="coordinate_rmse_yards",
            y="model",
            orientation="h",
            color="feature_gain_percent",
            title="Same estimator, different features",
            labels={"coordinate_rmse_yards": "Development RMSE (yards)"},
        )
    )
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    ax.barh(scores.model, scores.coordinate_rmse_yards, color="#247A89")
    ax.set_xlabel("Development coordinate RMSE (yards); lower is better")
    static(fig)
    display(Markdown(f"**Selected on inner folds:** {probe['selection']['selected_model']}"))
else:
    display(Markdown("The completed fixed-estimator report is not available in this checkout."))

## 2 · Does additional width still pay?

All candidates are screened on training data. The wider search preserves the 250-feature union, interleaves ranked families, and examines the full training-eligible pool. Additional columns pass a correlation check on 8,192 deterministic training rows at a 0.9995 threshold. The 512, 1,024, 2,048, 4,096, and 8,192 requested budgets are nested up to the eligible count.

A large candidate count is not a stopping argument. Examine incremental gains across all inner folds, paired development-game uncertainty, and computational cost. Selection pools squared errors over the three inner folds; it does not average fold RMSEs or select on development.

In [ ]:
if budget:
    widths = pd.DataFrame(
        [
            {
                "fold": fold["fold"]["name"],
                "model": row["model"],
                "feature_count": row["feature_count"],
                "coordinate_rmse_yards": row["coordinate_rmse_yards"],
            }
            for fold in [*budget["inner_folds"], budget]
            for row in fold["models"]
        ]
    )
    display(widths.pivot(index="model", columns="fold", values="coordinate_rmse_yards").round(5))
    display(pd.Series(budget["inner_scores"], name="Pooled inner-fold RMSE").sort_values())
    interactive(
        px.line(
            widths,
            x="feature_count",
            y="coordinate_rmse_yards",
            color="fold",
            markers=True,
            title="Marginal value of a wider screened representation",
            labels={"feature_count": "Retained features", "coordinate_rmse_yards": "RMSE (yards)"},
        )
    )
    fig, ax = plt.subplots(figsize=(9, 4.5), layout="constrained")
    for name, rows in widths.groupby("fold", sort=True):
        ax.plot(rows.feature_count, rows.coordinate_rmse_yards, marker="o", label=name)
    ax.set_xlabel("Retained features")
    ax.set_ylabel("Coordinate RMSE (yards)")
    ax.legend()
    static(fig)
    if attribution:
        display(pd.DataFrame(attribution["development"]["incremental_width"]).round(5))
else:
    display(Markdown("Width comparisons are incomplete until every fold has a verified result."))

## 3 · Which families help, and which disappoint?

Strict removals delete a family from the 250-feature union, refit with identical settings, and add no replacement columns. A positive RMSE change indicates useful conditional information. The positional linear fallback uses another estimator and is reported separately.

Reachability and role-specific destination geometry were consistently useful in the chronological experiments. Route descriptors, role-gated terms, and metadata showed more mixed or small effects. A weak sequential linear correction can still help a nonlinear model; every conclusion must identify its comparator.

In [ ]:
if ablation:
    rows = []
    for fold in [*ablation["inner_folds"], ablation]:
        reference = next(
            r["coordinate_rmse_yards"] for r in fold["models"] if r["model"] == "all_engineered"
        )
        for row in fold["models"]:
            if row["model"].startswith("without_"):
                rows.append(
                    {
                        "fold": fold["fold"]["name"],
                        "removed": row["model"].removeprefix("without_"),
                        "rmse_change": row["coordinate_rmse_yards"] - reference,
                    }
                )
    removals = pd.DataFrame(rows)
    display(removals.pivot(index="removed", columns="fold", values="rmse_change").round(5))
    interactive(
        px.bar(
            removals,
            x="removed",
            y="rmse_change",
            color="fold",
            barmode="group",
            title="Strict family removals; fixed estimator",
            labels={"rmse_change": "RMSE increase after removal (yards)"},
        )
    )
    display(
        pd.DataFrame(ablation["models"])[["model", "feature_count", "coordinate_rmse_yards"]].round(
            5
        )
    )

### Explain the wider representation separately

Family permutations shuffle complete trajectories within role and forecast horizon, preserve exact forecast-frame alignment, and hold the physical baseline fixed. Five seeds measure shuffle sensitivity. Their range is **not a confidence interval**. Permutation measures reliance by this model, not causal importance; correlated families may substitute for each other.

Fresh fixed-capacity refits remove metadata/player history and then all optional metadata/telemetry dependencies. These check whether the wider model remains strong when potentially unavailable feature families are excluded.

In [ ]:
if attribution:
    display(
        Markdown(
            f"**Inner-fold choice:** {attribution['selected_model']} · "
            f"{attribution['development']['feature_count']:,} retained columns"
        )
    )
    dependence = pd.DataFrame(attribution["development"]["permutation"])
    display(
        dependence.drop(columns="seed_changes")
        .sort_values("rmse_increase_mean", ascending=False)
        .round(5)
    )
    interactive(
        px.bar(
            dependence.sort_values("rmse_increase_mean"),
            x="rmse_increase_mean",
            y="family",
            orientation="h",
            title="Conditional reliance of the selected wide representation",
            labels={"rmse_increase_mean": "Development RMSE increase (yards)"},
        )
    )
    figure = ROOT / "artifacts/feature_attribution/figure.png"
    if not figure.exists():
        figure = PUBLISHED / "feature_attribution.png"
    if figure.exists():
        display(Image(filename=str(figure)))
    display(
        pd.DataFrame(
            [
                {
                    "fold": fold["fold"],
                    **{
                        key: row[key]
                        for key in [
                            "model",
                            "feature_count",
                            "coordinate_rmse_yards",
                            "delta_vs_full_ci95",
                        ]
                    },
                }
                for fold in [*attribution["inner_folds"], attribution["development"]]
                for row in fold["omissions"]
            ]
        ).round(5)
    )

## 4 · Broad feature coverage with explicit prediction-time boundaries

The catalogs cover motion, landing geometry, lags, multiscale trends and volatility, robust path distributions, matched opponent/receiver histories, relative ranks, nonlinear and role-conditioned crosses, arrival feasibility, geometric player-set pools, role-specific destinations, and training-only route representations.

Player/role residual histories use **strictly earlier game dates**: the entire current date is excluded, smoothing supports sparse players, and evaluation histories are frozen from training. Route components and prototypes are fitted separately inside each training fold. Post-throw coordinates, future game results, full-season target averages, and unverified external ratings cannot enter prediction inputs.

In [ ]:
if research:
    families = pd.DataFrame(research["families"])
    display(families[["stage", "family", "candidates", "nonconstant", "constant_or_near_constant"]])
    display(
        Markdown(
            f"**{research['candidate_features']:,} candidates; "
            f"{int(families.nonconstant.sum()):,} nonconstant** on development-training rows. "
            "Eligibility alone does not prove predictive value."
        )
    )
    interactive(
        px.bar(
            families,
            x="candidates",
            y="family",
            color="stage",
            orientation="h",
            title="Candidate families and search coverage",
        )
    )
    display(
        pd.DataFrame(
            [
                {
                    "stage": row["stage"],
                    "model": row["model"],
                    "all_three_folds": len(row["all_three_folds"]),
                    "any_fold": row["selected_in_any_fold"],
                    "pairwise_jaccard": row["pairwise_jaccard"],
                }
                for row in research["stability"]
            ]
        )
    )

### Stability can be strong at the family level and weak at the column level

Combined context improved all three chronological folds. Individual selected columns overlap less because many lagged and gated variables express related signals. We report that overlap. PCA names do not guarantee identical axes across folds. This study has one labelled season: it does not establish across-season stability.

The sequential 186-feature predictor is an interpretable intermediate. Its permutation report explains that fit, not the current joint linear model or the wider tree.

In [ ]:
if research:
    fold_results = pd.DataFrame(research["fold_results"])
    chosen = fold_results.loc[
        ((fold_results.stage == "research") & (fold_results.model == "plus_balanced"))
        | ((fold_results.stage == "context") & (fold_results.model == "plus_context"))
        | ((fold_results.stage == "representation") & (fold_results.model == "plus_representation"))
    ]
    display(
        chosen[
            [
                "stage",
                "fold",
                "training_games",
                "validation_games",
                "coordinate_rmse_yards",
                "improvement_vs_parent_percent",
            ]
        ].round(5)
    )
    if importance:
        display(
            pd.DataFrame(importance["rows"])[
                ["family", "retained_features", "mean_rmse_increase_yards", "shuffle_sd_yards"]
            ].round(5)
        )

## 5 · Carry the validated features into actual inference

The joint linear experiment first tests whether refitting all coefficients recovers signal lost by sequential corrections. The portable tree handoff then reuses the already fitted, fixed-capacity feature trees. It chooses between metadata-free availability profiles on the inner folds, converts their numerical splits to a standalone representation, and drops columns never used by a split. Prediction parity is required after this lossless pruning. No additional model tuning occurs.

Raw inference must reproduce the cached-feature score on **every development frame**. Stress scenarios remove metadata, remove telemetry, and clear player history. Missing required fields select an independently fitted compatible profile. Silent zero filling is not a valid availability fallback.

In [ ]:
if joint:
    display(
        pd.DataFrame(joint["models"])[["model", "feature_count", "coordinate_rmse_yards"]].round(5)
    )
    display(pd.Series(joint["inner_scores"], name="Pooled inner-fold RMSE").sort_values())
    display(Markdown(f"**Selected linear profile:** {joint['selected_model']}"))
if inference:
    display(pd.DataFrame(inference["scenarios"]).round(5))
    display(
        Markdown(
            f"**Actual raw-input predictor:** {inference['selected_stage']} / "
            f"{inference['selected_model']} · {inference['retained_features']} features. "
            "Publication verifies source hashes, fitted artifacts, and per-week replay receipts."
        )
    )
else:
    display(Markdown("Full raw-input replay must pass before this path is called validated."))

### Inspect one play without selecting a flattering example

The example is the first development play by game/play identifier, showing up to three requested players in identifier order. Selection does not use prediction error. Solid dots mark the throw. The embedded figure uses the current validated research predictor. Regenerating this particular figure requires private tracking; the full quantitative comparisons above also work from the published aggregates.


In [ ]:
if inference and (ROOT / "artifacts/research/inference/model.json").exists():
    import numpy as np

    from nfl_trajectory.feature_experiment import load_week
    from nfl_trajectory.feature_research import verify_inputs
    from nfl_trajectory.motion import KEYS
    from nfl_trajectory.research_inference import predict_research, research_bundle

    fitted = research_bundle(ROOT)
    caches, _ = verify_inputs(ROOT)
    for cache in caches:
        _, requests, arrays = load_week(cache)
        available = requests.loc[requests.game_id.isin(fitted["evaluation_games"])]
        if available.empty:
            continue
        game, play = available[["game_id", "play_id"]].sort_values(["game_id", "play_id"]).iloc[0]
        chosen = requests.game_id.eq(game) & requests.play_id.eq(play)
        request = requests.loc[chosen, KEYS].reset_index(drop=True)
        truth = request.assign(x=arrays["truth"][chosen, 0], y=arrays["truth"][chosen, 1])
        observed = pd.read_csv(ROOT / "data/raw/train" / (cache.parent.name + ".csv"))
        observed = observed.loc[observed.game_id.eq(game) & observed.play_id.eq(play)]
        predicted = predict_research(observed, request, fitted)
        fig, ax = plt.subplots(figsize=(10, 5.5), layout="constrained")
        cloud = []
        colors = ["#247A89", "#D57838", "#8053A0"]
        for index, player in enumerate(sorted(request.nfl_id.unique())[:3]):
            history = observed.loc[observed.nfl_id.eq(player)].sort_values("frame_id").tail(15)
            actual = truth.loc[truth.nfl_id.eq(player)].sort_values("frame_id")
            estimate = predicted.loc[predicted.nfl_id.eq(player)].sort_values("frame_id")
            label = chr(ord("A") + index)
            for frame, kind, style in [
                (history, "observed", ":"),
                (actual, "truth", "-"),
                (estimate, "predicted", "--"),
            ]:
                ax.plot(
                    frame.x,
                    frame.y,
                    style,
                    color=colors[index],
                    linewidth=2,
                    label=f"Player {label}: {kind}",
                )
                cloud.append(frame[["x", "y"]].to_numpy())
            ax.scatter(history.x.iloc[-1], history.y.iloc[-1], color=colors[index], s=35)
        landing = observed[["ball_land_x", "ball_land_y"]].iloc[0].to_numpy(float)
        ax.scatter(*landing, marker="*", s=180, color="#C68A12", label="Supplied landing point")
        points = np.vstack([*cloud, landing[None, :]])
        ax.set_xlim(max(0, points[:, 0].min() - 4), min(120, points[:, 0].max() + 4))
        ax.set_ylim(max(0, points[:, 1].min() - 4), min(53.3, points[:, 1].max() + 4))
        ax.set_aspect("equal", adjustable="box")
        ax.set_facecolor("#F4F8F4")
        ax.grid(alpha=0.25)
        ax.set_xlabel("Field x (yards)")
        ax.set_ylabel("Field y (yards)")
        ax.set_title("Observed motion, truth, and the current research predictor")
        ax.legend(ncol=3, fontsize=8, loc="upper center", bbox_to_anchor=(0.5, -0.15))
        static(fig)
        display(
            pd.DataFrame(
                [
                    {
                        "Player": chr(ord("A") + i),
                        "Role": observed.loc[observed.nfl_id.eq(player), "player_role"].iloc[0],
                    }
                    for i, player in enumerate(sorted(request.nfl_id.unique())[:3])
                ]
            )
        )
        display(Markdown(f"**Displayed predictor:** {fitted['selected_model']}"))
        break
else:
    display(Markdown("The canonical executed notebook contains the private-data example figure."))

### Verify the organizer interface on its own unlabelled sample

The organizer sends one play at a time and expects finite x/y predictions in the requested order. The test uses checksum-verified organizer source and sample inputs, checks package/standalone parity for every callback, and verifies Parquet row identifiers. Output stays in the isolated quality directory.

A pass establishes interface compatibility. It gives no accuracy score, across-season performance result, or competition submission.

In [ ]:
if gateway:
    display(
        pd.DataFrame(
            [
                {
                    key: gateway[key]
                    for key in [
                        "official_gateway_status",
                        "selected_model",
                        "retained_features",
                        "sample_rows",
                        "plays",
                        "sample_seasons",
                        "callback_seconds_median",
                        "callback_seconds_p95",
                        "callback_seconds_max",
                        "standalone_prediction_parity",
                        "labels_available",
                        "competition_submission",
                    ]
                }
            ]
        )
    )
else:
    display(Markdown("Organizer gateway validation is not yet recorded in this checkout."))

## 6 · Preserve the original research lesson

Landing ridge initially reached 0.9269 RMSE versus 0.9896 for role-conditioned motion. The broader interaction challenger replaced 23 of the 64 landing columns. Its weaker score confounded adding interactions with removing useful terms. This motivated nested additions, chronological inner selection, and strict removals.

These original diagnostics explain the research design; they are not the latest performance claim.

In [ ]:
from nfl_trajectory.research import error_budget

display(
    pd.DataFrame(feature_summary["models"])[
        [
            "model",
            "selected_features",
            "coordinate_rmse_yards",
            "ade_frame_weighted_yards",
            "fde_trajectory_weighted_yards",
            "p95_displacement_yards",
        ]
    ].round(4)
)
display(
    pd.DataFrame(
        {
            "Removed landing feature": selection["removed_from_landing"],
            "Added interaction feature": selection["added_by_interaction"],
        }
    )
)
roles = error_budget(feature_summary, "role")
horizons = error_budget(feature_summary, "forecast_second")
display(roles.round(3))
display(horizons.round(3))
interactive(
    px.bar(
        horizons,
        x="value",
        y="squared_error_share_percent",
        title="Original landing-model error budget",
        labels={
            "value": "Forecast second",
            "squared_error_share_percent": "Squared error share (%)",
        },
    )
)

## 7 · Feature engineering is a completion gate

The official metric is

$$\mathrm{RMSE}=\sqrt{\frac{\sum_{i=1}^{N}[(\hat{x}_i-x_i)^2+(\hat{y}_i-y_i)^2]}{2N}}.$$

Frames have equal coordinate weight. Game-cluster bootstraps preserve within-game dependence. Paired intervals describe uncertainty on these development games; they do not erase repeated research decisions or establish multi-season generalization.

Closing the gate requires justified coverage, training-only screening, leakage tests, fixed-estimator improvements, ablations, stability, robust inference, and diminished marginal gains from realistic extensions. External team/coaching ratings lack a verified prediction-time join here. Train-only route representations are implemented; a pretrained football foundation model is not.

**The gate stays open until width and robustness evidence supports closure.** The [research plan](../docs/RESEARCH_PLAN.md) records that decision. Final refitting and the reserved holdout come afterwards. The inference report identifies the actual exported profile and its verified metric; promotion requires raw and standalone parity.

## Generate and download your own inference artifact

The switch is off for review and automated publication. Enabling it verifies and exports the current local **research** bundle with its fitted availability fallback. It provides a download and never submits to Kaggle. The exporter loads the current source-verified research profile; it cannot silently advertise another experiment's score.

In [ ]:
import base64
import subprocess
import sys

from IPython.display import HTML

GENERATE_EXPORT = False
if GENERATE_EXPORT:
    local_summary = ROOT / "artifacts/research/summary.json"
    if not local_summary.is_file():
        raise FileNotFoundError("Verified local research artifacts are required for an export.")
    requested_model = "research"
    subprocess.run(
        [sys.executable, str(ROOT / "kaggle/export.py"), "--model", requested_model],
        cwd=ROOT,
        check=True,
    )
    export_path = ROOT / "artifacts/kaggle/submission.ipynb"
    payload = base64.b64encode(export_path.read_bytes()).decode("ascii")
    display(
        HTML(
            '<a download="submission.ipynb" href="data:application/x-ipynb+json;base64,'
            + payload
            + '">Download the inference notebook you generated</a>'
        )
    )
else:
    display(
        Markdown(
            "**Export is off.** Set `GENERATE_EXPORT = True` "
            "to build and download your own artifact."
        )
    )